# Calibración del catálogo

El profesor corre esto **una vez** en Colab (T4, `qwen17b`) antes de soltar el catálogo. Comprueba que la precisión sube de a poco al ir fijando ranuras, y que el techo se queda cerca de 30%.

No es el notebook del curso.

Cómo leer lo que salga:

- Si una opción casi no baja la precisión, el texto no está haciendo efecto y hay que endurecerlo.
- Si una opción deja la precisión en cero, el efecto es demasiado ancho: tiene que apuntar a una familia más chica.
- Si el ascenso por coordenadas llega al óptimo, el ejercicio está fácil y alguna ranura necesita una interacción más fuerte.


## 1 · Instalar y bajar los 3 archivos  ·  *al terminar, reinicia el entorno de ejecución*

Baja `oraculo.py`, `ayudas.py` y `datos_visibles.json`, más las librerías que usan el modelo y los verificadores. El reinicio es obligatorio: si siguen sin reiniciar, `transformers` a veces queda a medias y `cargar_modelo` falla con un error opaco.


In [ ]:
# bitsandbytes: checkpoints 4-bit (qwen8b, mistral7b). nltk/spacy/emoji/langdetect:
# los usa el verificador de open-instruct, no el notebook.
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!git clone -q https://github.com/allenai/open-instruct

REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-AyD-algoritmos/main"
# --no-cache: Colab a veces reusa un ayudas.py viejo y el alias no existe.
!wget -q --no-cache -O oraculo.py {REPO}/oraculo.py
!wget -q --no-cache -O ayudas.py {REPO}/ayudas.py
!wget -q --no-cache -O datos_visibles.json {REPO}/datos_visibles.json

# Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")


## 2 · El modelo

Antes: **Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU**.
Sin GPU, `cargar_modelo` no arranca.

`cargar_modelo` acepta un alias de la tabla o cualquier id público de Hugging Face (`org/nombre`). En `ayudas.py` hay más alias (`qwen8b`, `mistral7b`): caben en T4 en 4-bit, pero cada consulta tarda más.

| Alias | Checkpoint | Tamaño | Notas |
|---|---|---|---|
| `qwen17b` | `Qwen/Qwen3-1.7B` | 1.7B | el de por defecto; el más rápido |
| `ministral3b` | `mistralai/Ministral-3-3B-Instruct-2512-BF16` | 3.8B | fp16, ~7.7 GB de VRAM |
| `llama3b` | `unsloth/Llama-3.2-3B-Instruct` | 3.2B | fp16, ~6.4 GB de VRAM |

Son tres familias distintas (Qwen, Mistral, Llama): sirve para ver si su configuración generaliza o si solo le funciona a un modelo.

El caché guarda el nombre del modelo en la clave, así que cambiar de modelo **no** reusa respuestas del anterior: vuelve a gastar rollouts.

> Usen el repo `-BF16` de Ministral 3. El repo por defecto es FP8 y la T4 no lo soporta.


In [ ]:
from ayudas import cargar_modelo

# Alias → checkpoint. El nombre entra en la clave de caché: si cambian de
# modelo, las respuestas anteriores no se reusan.
#  "qwen17b"     → Qwen/Qwen3-1.7B
#  "ministral3b" → mistralai/Ministral-3-3B-Instruct-2512-BF16
#  "llama3b"     → unsloth/Llama-3.2-3B-Instruct
modelo = cargar_modelo("qwen17b")


## 3 · El oráculo

`dividir` parte `datos_visibles.json` en búsqueda (150) y validación (300). Busquen solo sobre `busqueda`. `validar` mide la config ya elegida sobre instancias que no se usaron al buscar: sirve para ver si generaliza a **instancias** nuevas, no a tipos de restricción nuevos.

Si Colab se desconecta, descomenten las dos líneas de Drive para no perder el caché.


In [ ]:
# Descomenten estas dos líneas si quieren que el caché sobreviva a una
# desconexión: el path de Drive reemplaza cache_oraculo.json local.
# from google.colab import drive
# drive.mount("/content/drive")

from oraculo import Oraculo, CATALOGO, espacio
from ayudas import cargar_datos, dividir

datos = cargar_datos()
busqueda, validacion = dividir(datos)
# 1.7B en fp16 deja VRAM en la T4: un lote más grande genera más prompts a la vez.
oraculo = Oraculo(modelo, busqueda, validacion, lote=16, presupuesto=32_000_000)
# oraculo = Oraculo(modelo, busqueda, validacion, cache="/content/drive/MyDrive/oraculo_cache.json", lote=16, presupuesto=32_000_000)

# 1024 configs (temperatura 0.0). Para incluir 0.3 y 0.7: espacio(TEMPERATURAS)
CONFIGS = espacio()
print(len(CONFIGS), "configuraciones posibles")


## 4 · Barrido por ranura

Para cada ranura se prueban las cuatro opciones con las otras cuatro en el último índice, sobre las mismas 40 instancias del notebook del curso. La precisión y la familia de `violo` más repetida dicen qué efecto colateral está pegando de verdad.


In [ ]:
from collections import Counter

from oraculo import CATALOGO, RANURAS

# Mismo lote que el notebook del curso: 40 instancias, paso de 2.5%.
INSTANCIAS = busqueda[:40]
SEMILLA = 1
CONSULTAS = {"n": 0}


def indice_final(ranura):
    """Última opción de la ranura: la que no trae el efecto colateral."""
    return len(CATALOGO[ranura]) - 1


def config_techo():
    config = {ranura: indice_final(ranura) for ranura in RANURAS}
    config["temperatura"] = 0.0
    return config


def medir(config):
    CONSULTAS["n"] += 1
    return oraculo.evaluar(config, INSTANCIAS, semilla=SEMILLA)


def violo_frecuente(trazas):
    """Familia que más se repite en los fallos: es lo que esa opción está rompiendo."""
    conteo = Counter(t["violo"] for t in trazas if t.get("violo"))
    if not conteo:
        return "—"
    familia, n = conteo.most_common(1)[0]
    return f"{familia} ({n})"


print(f"lote: {len(INSTANCIAS)} instancias\n")
print("=== Barrido: una ranura a la vez, las otras en el último índice ===")
for ranura in RANURAS:
    print(f"\n## {ranura}")
    for i, texto in enumerate(CATALOGO[ranura]):
        config = config_techo()
        config[ranura] = i
        r = medir(config)
        marca = "  ← último" if i == indice_final(ranura) else ""
        primera = texto.splitlines()[0]
        print(f"  [{i}] {r.precision:6.1%}  violo: {violo_frecuente(r.trazas)}{marca}")
        print(f"       {primera[:90]}")


## 5 · Extremos

Todo índice 0 es el piso (esperado cerca de 2–8%). Todo último índice es el techo (esperado cerca de 25–35%). Si el piso es 0% o el techo se va mucho más arriba, el catálogo no es el del curso.


In [ ]:
piso = {ranura: 0 for ranura in RANURAS}
piso["temperatura"] = 0.0
techo = config_techo()

r_piso = medir(piso)
r_techo = medir(techo)
print(f"piso   (todo índice 0): {r_piso.precision:.1%}   esperado ~2–8%")
print(f"techo  (todo último):   {r_techo.precision:.1%}   esperado ~25–35%")
print("piso ", piso)
print("techo", techo)


## 6 · Ascenso por coordenadas

Arranca en todo índice 0. Ranura por ranura prueba los cuatro índices y se queda con el que más sube. Otra vuelta solo si alguna ranura todavía mejora. La curva es la precisión después de cada cambio. Al final se compara la meseta con el techo de la celda anterior.


In [ ]:
from ayudas import curva


def ascenso():
    """Sube una ranura a la vez desde todo índice 0.

    En cada ranura se prueban los otros índices y se adopta el mejor.
    Una vuelta que no cambia ninguna ranura es la meseta.
    """
    config = {ranura: 0 for ranura in RANURAS}
    config["temperatura"] = 0.0
    gastadas = 0
    r = medir(config)
    gastadas += 1
    puntos = [r.precision]
    print(f"inicio  {r.precision:6.1%}  {config}")
    while True:
        mejoro = False
        for ranura in RANURAS:
            mejor_i = config[ranura]
            mejor_p = puntos[-1]
            for i in range(len(CATALOGO[ranura])):
                if i == config[ranura]:
                    continue
                prueba = dict(config)
                prueba[ranura] = i
                rr = medir(prueba)
                gastadas += 1
                if rr.precision > mejor_p:
                    mejor_p = rr.precision
                    mejor_i = i
            if mejor_i != config[ranura]:
                config[ranura] = mejor_i
                puntos.append(mejor_p)
                mejoro = True
                print(f"  {ranura} → {mejor_i}  {mejor_p:6.1%}")
        if not mejoro:
            print("meseta: una vuelta completa sin subida")
            break
    return config, puntos, gastadas


config_greedy, curva_greedy, evals_greedy = ascenso()
optimo = config_techo()
print()
print("meseta en", {ranura: config_greedy[ranura] for ranura in RANURAS}, f"{curva_greedy[-1]:.1%}")
print(
    "óptimo (todo último índice):",
    {ranura: optimo[ranura] for ranura in RANURAS},
    f"{r_techo.precision:.1%}",
)
print("¿llegó al óptimo?", "sí" if config_greedy == optimo else "no")
curva(curva_greedy)


## 7 · Resumen

Piso, techo, meseta del ascenso y cuántas consultas gastó. El caché no vuelve a generar lo ya medido en el barrido; el conteo igual anota cada llamada a `evaluar`.

Si el piso quedó en cero, alguna opción es demasiado ancha. Si el techo está lejos de 25–35%, el catálogo no tiene el margen del curso. Si la meseta coincide con el óptimo, falta una interacción entre ranuras: arreglar una no debería alcanzar para llegar.


In [ ]:
filas = [
    ("piso", f"{r_piso.precision:.1%}"),
    ("techo", f"{r_techo.precision:.1%}"),
    ("meseta greedy", f"{curva_greedy[-1]:.1%}"),
    ("evaluaciones greedy", str(evals_greedy)),
    ("evaluaciones del notebook", str(CONSULTAS["n"])),
    ("llegó al óptimo", "sí" if config_greedy == optimo else "no"),
]
ancho = max(len(nombre) for nombre, _ in filas)
for nombre, valor in filas:
    print(f"{nombre:<{ancho}}  {valor}")
print("config greedy", {ranura: config_greedy[ranura] for ranura in RANURAS})
